# Computer Vision Notebook 3: Classic CNN on MNIST Digits

**Goal:** train a small CNN on handwritten digit images.

This notebook connects to the classic line of digit recognition work that helped popularize convolutional neural networks. The model is LeNet-inspired: convolution, pooling, convolution, pooling, dense layers, and a 10-class output.

The dataset is small enough for Colab and is a good first neural-network demo.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print('TensorFlow version:', tf.__version__)

## 1. Load MNIST

Each image is 28 x 28 pixels. The labels are digits 0 through 9.

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# CNNs expect a channel dimension: 28 x 28 x 1
x_train = x_train[..., np.newaxis]
x_test = x_test[..., np.newaxis]

print('Training images:', x_train.shape)
print('Test images:', x_test.shape)

In [ ]:
plt.figure(figsize=(8, 3))
for i in range(10):
    plt.subplot(2, 5, i+1)
    plt.imshow(x_train[i, :, :, 0], cmap='gray')
    plt.title(f'label: {y_train[i]}')
    plt.axis('off')
plt.tight_layout()

## 2. Build a small LeNet-style CNN

A CNN learns small filters that respond to useful visual patterns such as strokes, corners, and curves.

In [ ]:
model = keras.Sequential([
    layers.Input(shape=(28, 28, 1)),
    layers.Conv2D(6, kernel_size=5, activation='relu', padding='same'),
    layers.AveragePooling2D(pool_size=2),
    layers.Conv2D(16, kernel_size=5, activation='relu'),
    layers.AveragePooling2D(pool_size=2),
    layers.Flatten(),
    layers.Dense(120, activation='relu'),
    layers.Dense(84, activation='relu'),
    layers.Dense(10, activation='softmax')
])

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

## 3. Train the model

For a live workshop, 1 epoch is enough to demonstrate the process. For better performance, increase to 3-5 epochs.

In [ ]:
history = model.fit(x_train, y_train, epochs=1, batch_size=128,
                    validation_split=0.1, verbose=1)

In [ ]:
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f'Test accuracy: {test_acc:.3f}')

## 4. Visualize predictions

Green titles mean correct predictions; red titles mean mistakes. The model is not just memorizing pixels: it learns filters useful for recognizing patterns.

In [ ]:
probs = model.predict(x_test[:25], verbose=0)
preds = np.argmax(probs, axis=1)

plt.figure(figsize=(10, 10))
for i in range(25):
    plt.subplot(5, 5, i+1)
    plt.imshow(x_test[i, :, :, 0], cmap='gray')
    title = f'pred {preds[i]} / true {y_test[i]}'
    plt.title(title, color=('green' if preds[i] == y_test[i] else 'red'), fontsize=9)
    plt.axis('off')
plt.tight_layout()

## 5. Challenge

Change the number of filters, add one more convolution layer, or train for more epochs. How does accuracy change?